# 00 数据准备

从内嵌样本加载测试用例，自动生成反事实变体和掩码版本。

## 实验概述

**目的：** 为后续 notebook（01-06）准备标准化的测试数据，包括原始新闻、反事实变体和掩码版本。

**数据来源：** 42 条 A 股重大新闻事件（2020-2024），涵盖政策、企业、行业、宏观四大类别。

**两种反事实变体（用于泄露检测）：**
- `reverse_outcome`：翻转核心结论方向（利好↔利空）。非泄露模型应翻转预测，PC 高 = 泄露。
- `alter_numbers`：大幅修改数值至可能改变结论方向。非泄露模型应改变判断，PC 高 = 泄露。

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import set_seed
from src.models import *
from src.llm_client import LLMClient
from src.news_loader import load_test_cases, save_counterfactual_variants
from src.masking import generate_counterfactuals_batch, apply_masking
from src.display_utils import show_comparison, show_llm_response
import json

set_seed()
(PROJECT_ROOT / "data" / "results").mkdir(parents=True, exist_ok=True)

## 1. 加载测试用例

In [ ]:
test_cases = load_test_cases()
print(f"Loaded {len(test_cases)} test cases")
for tc in test_cases[:10]:
    print(f"  [{tc.id}] {tc.news.title} -> {tc.expected_direction.value}")

## 2. 自动生成反事实变体

对每条新闻生成两种反事实变体：
- **reverse_outcome**: 反转核心结论方向（测试模型是否依赖记忆而非文本分析）
- **alter_numbers**: 大幅修改数值至可能改变结论方向（测试模型对数量信息的敏感度）

共生成 42×2 = 84 个反事实变体。

In [ ]:
client = LLMClient()

# 并发生成所有反事实变体 (42 cases × 2 types = 84 个任务，线程池并发)
print(f"并发生成 {len(test_cases) * len(list(VariantType))} 个反事实变体 ...")
all_variants = generate_counterfactuals_batch(client, test_cases, validate=True)

# 按 case 分组打印结果
for tc in test_cases:
    case_variants = [v for v in all_variants if v.original_case_id == tc.id]
    print(f"[{tc.id}] {tc.news.title[:30]}...")
    for v in case_variants:
        status = v.modification_description or "ok"
        print(f"  {v.variant_type}: {status}")

print(f"\nTotal variants: {len(all_variants)}")
save_counterfactual_variants(all_variants)
print("Saved to data/generated/counterfactual_variants.json")

## 3. 预览反事实变体

In [ ]:
import random

# Side-by-side comparison for each variant type (random sample, rerun to refresh)
tc_map = {tc.id: tc for tc in test_cases}
for vt in VariantType:
    candidates = [v for v in all_variants if v.variant_type == vt.value]
    if candidates:
        example = random.choice(candidates)
        orig_tc = tc_map.get(example.original_case_id)
        orig_text = orig_tc.news.content if orig_tc else "(original not found)"
        show_comparison(orig_text, example.modified_content, title=f"变体类型: {vt.value} ({example.original_case_id})")

## 4. 生成掩码版本预览

展示各维度掩码效果，包括 rule-based 和 LLM-based 两种模式。

In [ ]:
from src.masking import mask_year, mask_entity, mask_numbers, mask_sector, apply_masking, MaskingConfig

# Random sample, rerun to refresh
sample = random.choice(test_cases)
text = sample.news.content
print(f"样本: [{sample.id}] {sample.news.title}")

# Rule-based masking comparisons
show_comparison(text, mask_year(text), title="年份掩码 (rule-based)")
show_comparison(text, mask_entity(text, sample.key_entities), title="实体掩码 (rule-based)")
show_comparison(text, mask_numbers(text), title="数字掩码 (rule-based)")
show_comparison(text, mask_sector(text), title="板块掩码 (rule-based)")

full_config = MaskingConfig(mask_year=True, mask_entity=True, mask_numbers=True, mask_sector=True)
show_comparison(text, apply_masking(text, full_config, sample.key_entities, client=client), title="全掩码 (rule-based)")

# LLM-based masking comparisons
llm_config_year = MaskingConfig(mask_year=True, mask_mode="llm")
llm_config_entity = MaskingConfig(mask_entity=True, mask_mode="llm")
llm_config_numbers = MaskingConfig(mask_numbers=True, mask_mode="llm")
llm_config_sector = MaskingConfig(mask_sector=True, mask_mode="llm")
llm_config_full = MaskingConfig(mask_year=True, mask_entity=True, mask_numbers=True, mask_sector=True, mask_mode="llm")

show_comparison(text, apply_masking(text, llm_config_year, sample.key_entities, client=client), title="年份掩码 (LLM-based)")
show_comparison(text, apply_masking(text, llm_config_entity, sample.key_entities, client=client), title="实体掩码 (LLM-based)")
show_comparison(text, apply_masking(text, llm_config_numbers, sample.key_entities, client=client), title="数字掩码 (LLM-based)")
show_comparison(text, apply_masking(text, llm_config_sector, sample.key_entities, client=client), title="板块掩码 (LLM-based)")
show_comparison(text, apply_masking(text, llm_config_full, sample.key_entities, client=client), title="全掩码 (LLM-based)")

## 5. 数据统计

In [ ]:
import pandas as pd

stats = []
for tc in test_cases:
    stats.append({
        "id": tc.id,
        "category": tc.news.category.value,
        "year": tc.news.publish_time.year,
        "direction": tc.expected_direction.value,
        "sector": tc.sector,
        "n_entities": len(tc.key_entities),
        "n_numbers": len(tc.key_numbers),
    })

df = pd.DataFrame(stats)
print("测试用例分布：")
print(f"\n按类别：\n{df['category'].value_counts()}")
print(f"\n按年份：\n{df['year'].value_counts().sort_index()}")
print(f"\n按方向：\n{df['direction'].value_counts()}")
print(f"\n按板块：\n{df['sector'].value_counts()}")